In [ ]:
from adaptive_latents import datasets
import numpy as np
import matplotlib.pyplot as plt
from adaptive_latents.pro_pls import proPLS
from adaptive_latents import CenteringTransformer, Pipeline, proSVD, KernelSmoother, sjPCA, mmICA, AnimationManager, ArrayWithTime, Concatenator
from tqdm.notebook import tqdm



In [ ]:
d = datasets.Leventhal24uDataset(bin_size=.1)

In [ ]:
d.trial_data.columns
# d.trial_data.loc[:,['centerIn', 'centerOut']]
# d.trial_data.loc[:,['sideIn', 'sideOut']]

d.trial_data.foodClick = d.trial_data.foodClick.apply(lambda x: max(x) if hasattr(x,'__len__') else x)
d.trial_data['trialEnd'] = d.trial_data.loc[:,['centerIn', 'centerOut', 'tone', 'sideIn', 'sideOut', 'foodClick', 'foodRetrieval', 'wrong']].max(axis=1)

In [ ]:
beh = np.zeros((d.neural_data.shape[0], 4))
beh = ArrayWithTime(beh, d.neural_data.t)

for _, (start, end) in d.trial_data.loc[:, ['cueOn', 'trialEnd']].iterrows():
    s = (start < d.neural_data.t) & (d.neural_data.t < end) 
    beh[s,0] = 1
beh[beh[:,0]==0,1] = 1

for _, (start, end) in d.trial_data.loc[:, ['centerIn', 'centerOut']].iterrows():
    s = (start < d.neural_data.t) & (d.neural_data.t < end) 
    beh[s,1] = 1

for _, (start, end) in d.trial_data.loc[:, ['sideIn', 'sideOut']].iterrows():
    s = (start < d.neural_data.t) & (d.neural_data.t < end) 
    beh[s,2] = 1

In [ ]:
c.output_streams

In [ ]:
p = Pipeline([
    KernelSmoother(tau=8),
    CenteringTransformer(),
    # proPLS(k=2, log_level=0),

    c:=Concatenator(output_streams={0:0, 1:0}),
    proSVD(k=6, init_size=50),
])


start = 1238
video_time = 40
tq = tqdm(total=start+video_time)

with AnimationManager(fps=40) as am:
    output = []
    for i, (output_row, stream) in enumerate(p.streaming_run_on([d.neural_data, beh*5], return_output_stream=True)):
        if i < 100 or stream == 'skip':
            continue
        output.append(output_row)
        o = np.squeeze(output)

        next_t = p.mid_run_sources[0][0].current_sample_time()
        if len(o.shape) == 2 and start < next_t < start + video_time:
            am.axs[0,0].cla()
            am.axs[0,0].autoscale(True)
            
            am.axs[0,0].scatter(o[:,0], o[:,1], alpha=.999**np.arange(o.shape[0])[::-1], color='C0', linewidth=0, s=5)
            am.axs[0,0].plot(o[-10:,0], o[-10:,1], color='C1')
            am.axs[0,0].set_title(f'{i}')
            am.axs[0,0].axis('equal')

            # if np.abs(next_t - (d.trial_data['centerIn'])).min() < .5:
            #     x = (am.axs[0,0].transAxes + am.axs[0,0].transData.inverted()).transform((.98,.98))
            #     am.axs[0,0].autoscale(False)
            #     am.axs[0,0].scatter(*x, color='green', s=100)

            am.grab_frame()
        elif next_t > start + video_time:
            break
        tq.update(np.floor(next_t - tq.n).astype(int))

In [ ]:
latents = ArrayWithTime.from_list(output, squeeze_type='to_2d')

fig, ax = plt.subplots()
ax.plot(latents[:,0], latents[:,1])